# Funding-carry tracker — persistent vs transient

The arb scan flagged Hyperliquid perps at 20–33% annualized funding. But carry is only worth capturing if it **persists** — a 30% rate that flips sign tomorrow nets ~nothing (and costs fees to rebalance). This ranks coins by **realized static carry** (the mean funding a delta-neutral hold actually collects) AND **sign-stability** (how often funding keeps its direction). The sweet spot is high carry *and* high stability. 30d hourly funding from `hl_funding_hist` (`data/defi.duckdb`).

In [ ]:
from pathlib import Path
import duckdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import os as _os
if _os.getenv('NB_DARK'):
    sns.set_theme(style='darkgrid', rc={
        'figure.facecolor':'#0d1117','axes.facecolor':'#161b22','savefig.facecolor':'#0d1117',
        'axes.edgecolor':'#30363d','axes.labelcolor':'#c9d1d9','text.color':'#c9d1d9',
        'axes.titlecolor':'#c9d1d9','xtick.color':'#8b949e','ytick.color':'#8b949e',
        'grid.color':'#21262d'})
else:
    sns.set_theme(style='whitegrid')
ROOT = Path.cwd()
while not (ROOT/'data').exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
con = duckdb.connect(str(ROOT/'data'/'defi.duckdb'), read_only=True)
f = con.execute('SELECT coin, t, funding_rate FROM hl_funding_hist ORDER BY coin, t').df()
con.close()
f['apr'] = f.funding_rate * 24 * 365 * 100   # annualized %
print(f'{len(f)} hourly funding obs across {f.coin.nunique()} coins, {f.t.min()} -> {f.t.max()}')

## Persistence metrics per coin

`carry` = mean annualized funding (what a fixed-direction delta-neutral hold collects). `stability` = % of hours funding keeps the sign of its mean. `vol` = std of annualized funding. `current` = latest reading (is today's number typical or a spike?).

In [ ]:
rows=[]
for coin, g in f.groupby('coin'):
    if len(g) < 100: continue
    a = g.apr.to_numpy(); m = a.mean()
    stability = float(np.mean(np.sign(a) == np.sign(m))) if m != 0 else 0.0
    rows.append({'coin':coin,'n':len(g),'carry_%':round(m,1),'stability':round(stability,2),
                 'vol_%':round(a.std(),1),'current_%':round(a[-1],1)})
d = pd.DataFrame(rows)
# 'quality' = realized carry scaled by how reliably it holds its sign
d['persistent_carry_%'] = (d['carry_%'].abs() * d['stability']).round(1)
d.sort_values('persistent_carry_%', ascending=False).head(15).reset_index(drop=True)

## The sweet spot: carry size vs sign-stability

Upper-right = big *and* reliable (worth capturing). Lower-right = big but flippy (a trap — high instantaneous funding that won't hold). Color = current reading.

In [ ]:
plt.figure(figsize=(10,6))
sc=plt.scatter(d['carry_%'].abs(), d['stability'], s=80, c=d['current_%'].abs(), cmap='viridis')
for _,r in d.iterrows():
    if abs(r['carry_%'])>8 or r['stability']>0.85:
        plt.annotate(r['coin'], (abs(r['carry_%']), r['stability']), fontsize=8, alpha=.8)
plt.colorbar(sc, label='|current funding| %/yr')
plt.xlabel('|mean carry| %/yr'); plt.ylabel('sign stability'); plt.axhline(0.8,color='crimson',lw=.6,ls='--')
plt.title('Funding carry: size vs persistence'); plt.show()

## Funding over time — top persistent carries

Does the funding actually hold, or is the high mean a spike?

In [ ]:
top = d.sort_values('persistent_carry_%', ascending=False).head(6).coin.tolist()
plt.figure(figsize=(12,5))
for coin in top:
    g = f[f.coin==coin]
    plt.plot(g.t, g.apr, lw=.8, label=coin)
plt.axhline(0,color='gray',lw=.6); plt.ylabel('funding (annualized %)'); plt.legend(ncol=3)
plt.title('Funding over time — top persistent-carry coins'); plt.show()

## Spike vs persistent: current reading ÷ 30-day mean

A coin whose *current* funding is far above its 30d mean is a spike (won't last); one near its mean is a stable carry.

In [ ]:
d2 = d[d['carry_%'].abs() > 5].copy()
d2['current_vs_mean'] = (d2['current_%'] / d2['carry_%']).round(2)
print('current ~ mean (≈1) = persistent; >>1 = today is a spike:')
d2.sort_values('persistent_carry_%', ascending=False)[['coin','carry_%','current_%','current_vs_mean','stability']].head(12).reset_index(drop=True)

## Verdict

This is how you separate a real carry from a headline number: rank by **mean** funding × **sign-stability**, not the instantaneous rate. A coin pinned at, say, +25%/yr with >90% sign-stability is a genuine delta-neutral yield; a coin showing +30% *today* but with low stability (funding that swings sign) is a trap — you'd pay funding on the flips and bleed fees rebalancing.

Honest caveats stand: capacity (the biggest, most-stable carries are often thin alts — size-limited), **spot-borrow cost** for the short-spot leg of negative-funding carries, perp/spot **basis risk**, and execution fees. The tracker tells you *which* carries are worth the trouble; it doesn't make them riskless. Refresh by re-running the funding backfill or letting `capture_hyperliquid` accumulate `hl_ctx`.